# Class Pollution Analysis Visualization

This notebook analyzes and visualizes the output.jsonl file containing class pollution function analysis results from the GitHub repository dataset.

In [12]:
import json

In [13]:
# Helper functions
def result_load(RESULT_PATH):
  """Load the result from the JSONL file."""
  with open(RESULT_PATH, 'r') as file:
    all_entries = []
    for line in file.readlines():
      try:
        entry = json.loads(line.strip())
        all_entries.append(entry)
      except json.JSONDecodeError:
        print(f"Error decoding JSON from line: {line.strip()}")
    return all_entries

def print_core_class_pollution(entry):
  """Print core class pollution functions from an entry in a nicely formatted way."""
  repo_name = entry.get('repo_name', 'Unknown Repository')
  repo_url = entry.get('repo_url', '')
  timestamp = entry.get('timestamp', '')
  
  # Get core class pollution functions
  all_functions = entry.get('class_pollution_functions', [])
  core_functions = [func for func in all_functions if func.get('core', False) and func.get('confirmed', False)]
  
  if not core_functions:
    print(f"No core class pollution functions found in {repo_name}")
    core_functions = all_functions
  
  
  print(f"\n{'='*80}")
  print(f"CORE CLASS POLLUTION FUNCTIONS - {repo_name}")
  print(f"{'='*80}")
  print(f"Repository: {repo_name}")
  print(f"URL: {repo_url}")
  print(f"Analyzed: {timestamp[:19] if timestamp else 'Unknown'}")
  print(f"Core Functions Found: {len(core_functions)}")
  
  for i, func in enumerate(core_functions, 1):
    print(f"\n{'-'*60}")
    print(f"Function #{i}: {func.get('func_name', 'Unknown')}")
    print(f"{'-'*60}")
    
    # Location information
    location = func.get('location', {})
    print(f"File: {location.get('file_path', 'Unknown')}")
    print(f"Line: {location.get('line_number', 'Unknown')}")
    
    # Pollution details
    print(f"Pollution Key: {func.get('pollution_key', 'Unknown')}")
    print(f"Pollution Value: {func.get('pollution_value', 'Unknown')}")
    print(f"Type: {func.get('type', 'Unknown')}")
    
    # Status indicators
    confirmed = "CONFIRMED" if func.get('confirmed', False) else "UNCONFIRMED"
    core = "CORE" if func.get('core', False) else "NON-CORE"
    print(f"Status: {confirmed} | {core}")
    
    # Description
    description = func.get('description', '')
    if description:
      print(f"Description: {description}")
    
    # POC if available
    poc = func.get('poc', '')
    if poc:
      print(f"PoC Verified: {func.get('poc_verified', 'Unknown')}")
      print(f"PoC Comment: {func.get('poc_comments', 'No comment')}")
      print(f"PoC Code:")
      # Show first few lines of PoC
      poc_lines = poc.split('\n')
      for line in poc_lines:
        print(f"{line}")

def count_verified_poc_with_original_import(results):
  """
  Count and display repositories with verified PoC that have original package imports.
  
  Args:
    results: List of result entries from JSONL file
    
  Returns:
    int: Count of verified PoCs with original package imports
  """
  verified_repos = []
  total_count = 0
  
  for entry in results:
    if not isinstance(entry, dict):
      continue
      
    repo_name = entry.get('repo_name', 'Unknown')
    repo_functions = []
    
    for func in entry.get('class_pollution_functions', []):
      if not isinstance(func, dict):
        continue
        
      # Check all required conditions
      is_core = func.get('core', False)
      is_confirmed = func.get('confirmed', False)
      has_original_import = func.get('original_package_import', False)
      
      if is_core and is_confirmed and has_original_import:
        total_count += 1
        func_name = func.get('func_name', 'Unknown function')
        repo_functions.append(func_name)
    
    # Only add repo if it has qualifying functions
    if repo_functions:
      verified_repos.append({
        'repo': repo_name,
        'functions': repo_functions,
        'count': len(repo_functions)
      })
  
  return verified_repos, total_count

def display_verified_poc_results(results):
  """Display the results in a nice format."""
  verified_repos, total_count = count_verified_poc_with_original_import(results)
  
  print("=" * 80)
  print("REPOSITORIES WITH VERIFIED POC AND ORIGINAL PACKAGE IMPORTS")
  print("=" * 80)
  
  if not verified_repos:
    print("No repositories found matching the criteria.")
    return total_count
  
  for i, repo_info in enumerate(verified_repos, 1):
    print(f"\n{i:2d}. Repository: {repo_info['repo']}")
    print(f"    Functions ({repo_info['count']}):")
    for func in repo_info['functions']:
      print(f"      -- {func}")
  
  print("\n" + "=" * 80)
  print(f"SUMMARY: {total_count} verified PoCs found across {len(verified_repos)} repositories")
  print("=" * 80)
  
  return total_count


In [14]:
display_verified_poc_results(RESULT)

REPOSITORIES WITH VERIFIED POC AND ORIGINAL PACKAGE IMPORTS

 1. Repository: glom
    Functions (3):
      -- _t_eval
      -- _glom
      -- _assign_op

 2. Repository: glom
    Functions (1):
      -- _t_eval

 3. Repository: glom
    Functions (2):
      -- _t_eval / _assign_op
      -- _assign_op

 4. Repository: Open-Assistant
    Functions (1):
      -- rsetattr

 5. Repository: pytorch-image-models
    Functions (1):
      -- set_layer

 6. Repository: spaCy
    Functions (1):
      -- __call__

 7. Repository: pytorch-lightning
    Functions (1):
      -- _set_module_by_path

 8. Repository: ColossalAI
    Functions (5):
      -- setattr_
      -- solution_annotation_pass
      -- size_value_converting_pass
      -- module_params_sharding_pass
      -- module_params_sharding_pass

 9. Repository: minGPT
    Functions (1):
      -- merge_from_args

10. Repository: diffusers
    Functions (2):
      -- create_quantized_param
      -- create_quantized_param

11. Repository: sd-web

228

In [10]:
RESULT = result_load("/home/jackfromeast/Desktop/python-class-pollution/tasks/llm-check/github-1K-r2/logs/output.jsonl")
RESULT[9]

Error decoding JSON from line: {"repo_name":"lorax","repo_url":"https://github.com/predibase/lorax","timestamp":"2025-08-20T03:42:32.366276","class_pollution_functions":[{"poc":"# CLASS POLLUTION PROOF OF CONCEPT (PoC)\n# Class Pollution Func: setdeepattr\n# Type: get-attr + set-attr (deepattr via dot notation)\n# NOTE: ORIGINAL PACKAGE IMPORT NOT POSSIBLE \n# Implementation follows original at /app/server/lorax_server/utils/gptq/quantize.py line 741\n# Unable to import due to unresolved internal dependencies and proto import errors in lorax_server package.\n\n# Minimal vulnerable function implementation:\ndef setdeepattr(module, full_name, tensor):\n    current = module\n    tokens = full_name.split(\".\")\n    for token in tokens[:-1]:\n        current = getattr(current, token)\n    setattr(current, tokens[-1], tensor)\n\n# Custom metaclass to allow changing __name__ for demonstration\nclass VillainMeta(type):\n    pass\nclass Victim(metaclass=VillainMeta):\n    pass\n\n# Preparation

{'repo_name': 'fairseq',
 'repo_url': 'https://github.com/facebookresearch/fairseq',
 'timestamp': '2025-08-19T07:18:04.535865',
 'class_pollution_functions': [{'poc': '# CLASS POLLUTION PROOF OF CONCEPT (PoC)\n# Class Pollution Func: _set_module_by_path\n# Type: get-attr-set-attr\n\n### Mimic implementation from fairseq/trainer.py\n# def _set_module_by_path(module, path, value):\n#     path = path.split(\'.\')\n#     for name in path[:-1]:\n#         module = getattr(module, name)\n#     setattr(module, path[-1], value)\n\ndef _set_module_by_path(module, path, value):\n    path = path.split(\'.\')\n    for name in path[:-1]:\n        module = getattr(module, name)\n    setattr(module, path[-1], value)\n\n### Preparation of target objects\nclass Dummy:\n    pass\n\nobj = Dummy()\nobj.data = Dummy()  # prepare a layer\n\n### Preparation of attacker-controlled pollution keys and values\npayload_path = \'data.__class__.__name__\'\npayload_value = \'pwnd\'\n\n# PoC harnesses\ndef run_poc()

In [16]:
x = [r for r in RESULT if r['repo_name'] == "docarray"][0]
print_core_class_pollution(x)  # Print the first entry as an example


CORE CLASS POLLUTION FUNCTIONS - docarray
Repository: docarray
URL: https://github.com/docarray/docarray
Analyzed: 2025-08-20T02:45:19
Core Functions Found: 1

------------------------------------------------------------
Function #1: __getitem__
------------------------------------------------------------
File: /app/docarray/data/torch_dataset.py
Line: 115
Pollution Key: attr (derived from keys in self._preprocessing, e.g., acc_path[-1])
Pollution Value: preprocess(value) (transformed value set on object attribute)
Type: get-attr-set-attr
Status: CONFIRMED | CORE
Description: Allows setting arbitrary attributes on contained objects using attacker-provided field.
PoC Verified: True
PoC Comment: This PoC shows that by supplying a field name of '__class__.__name__' in the preprocessing dictionary to MultiModalDataset from docarray, the __getitem__ function will set the __class__.__name__ attribute on the object, demonstrating two-layer class pollution. This is achieved by exploiting the 